# Penguins EDA

**Цель:** исследовать датасет пингвинов Palmer Archipelago, выявить различия между видами, островами и полом, найти ключевые морфологические зависимости.

**Структура ноутбука:**
1. Загрузка и первичный осмотр
2. Очистка данных
3. Распределения переменных
4. Анализ по категориям (пол, вид, остров)
5. Корреляционный анализ
6. Статистические тесты
7. Confounding variable: остров vs вид
8. Итоговые выводы

## 1. Загрузка и первичный осмотр

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu, spearmanr

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv('penguins.csv')

print('Размер:', df.shape)
print('\nПервые строки:')
df.head()

In [ ]:
print('Типы данных и пропуски:')
df.info()
print('\nДубликатов:', df.duplicated().sum())
print('\nУникальных видов:', df['species'].unique())
print('Уникальных островов:', df['island'].unique())

In [ ]:
print('Базовая статистика:')
df.describe(include='all')

**Наблюдения после первичного осмотра:**
- 344 строки, 9 столбцов, дубликатов нет
- 3 вида: Adelie, Chinstrap, Gentoo
- 3 острова: Biscoe, Dream, Torgersen
- Пропуски: `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g` — по 2 строки; `sex` — 11 строк
- Итого строк с пропусками нужно проверить

## 2. Очистка данных

In [ ]:
# Детальный анализ пропусков
print('Пропуски по столбцам:')
print(df.isnull().sum())
print(f'\nСтрок с пропусками: {df.isnull().any(axis=1).sum()}')
print(f'Процент от датасета: {df.isnull().any(axis=1).sum()/len(df)*100:.1f}%')

In [ ]:
# Пропусков 3.5% — удаляем строки (безопасно при <5%)
df_clean = df.dropna()

print(f'Строк до очистки:  {len(df)}')
print(f'Строк после:       {len(df_clean)}')
print(f'Удалено:           {len(df) - len(df_clean)} строк')

**Решение по очистке:**
- Пропусков ~3.5% — безопасно удалить через `dropna()`
- Если бы пропусков было >5%, правильнее заполнять медианой внутри каждого вида (не общей медианой), т.к. размеры Adelie и Gentoo сильно отличаются

## 3. Распределения переменных

In [ ]:
numeric_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for i, col in enumerate(numeric_cols):
    axes[0, i].hist(df_clean[col], bins=25, edgecolor='black', color='steelblue', alpha=0.7)
    axes[0, i].set_title(col, fontsize=9)

    axes[1, i].boxplot(df_clean[col], patch_artist=True,
                       boxprops=dict(facecolor='steelblue', alpha=0.6))
    axes[1, i].set_xticks([])
    axes[1, i].set_title(f'Boxplot: {col}', fontsize=9)

plt.suptitle('Распределения числовых переменных', fontsize=13)
plt.tight_layout()
plt.show()

**Наблюдения по распределениям:**
- `bill_length_mm` и `flipper_length_mm` — бимодальное распределение (два пика), что намекает на наличие подгрупп (виды)
- `body_mass_g` — также бимодальный, с правым хвостом (Gentoo тяжелее остальных)
- `bill_depth_mm` — наиболее равномерный
- Выбросов нет — все значения биологически разумны

## 4. Анализ по категориям

In [ ]:
# Распределение пингвинов по видам и островам
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_clean['species'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Количество пингвинов по видам')
axes[0].set_ylabel('Кол-во')
axes[0].tick_params(axis='x', rotation=0)

df_clean['island'].value_counts().plot(kind='bar', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Количество пингвинов по островам')
axes[1].set_ylabel('Кол-во')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Морфология по полу
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_clean.boxplot(column='body_mass_g', by='sex', ax=axes[0])
axes[0].set_title('Масса тела по полу')
axes[0].set_xlabel('Пол')
axes[0].set_ylabel('body_mass_g (g)')

df_clean.boxplot(column='flipper_length_mm', by='sex', ax=axes[1])
axes[1].set_title('Длина плавника по полу')
axes[1].set_xlabel('Пол')
axes[1].set_ylabel('flipper_length_mm (mm)')

plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# Морфология по виду — самый важный разрез
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    df_clean.boxplot(column=col, by='species', ax=axes[i])
    axes[i].set_title(f'{col} по виду')
    axes[i].set_xlabel('Вид')

plt.suptitle('')
plt.tight_layout()
plt.show()

**Наблюдения по категориям:**
- **Самцы крупнее самок** по массе и длине плавника — устойчивый паттерн
- **Gentoo** — самый крупный вид: масса ~5000g vs ~3700g у Adelie
- **Chinstrap** — самый длинный клюв (bill_length)
- **Adelie** — наибольшая глубина клюва (bill_depth) относительно длины

## 5. Корреляционный анализ

In [ ]:
# One-Hot Encoding для категориальных переменных
island_dummies = pd.get_dummies(df_clean['island'], prefix='island')
species_dummies = pd.get_dummies(df_clean['species'], prefix='species')
df_encoded = pd.concat([df_clean[numeric_cols], island_dummies, species_dummies], axis=1)

plt.figure(figsize=(12, 9))
sns.heatmap(df_encoded.corr(method='spearman'),
            annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Корреляционная матрица (Spearman)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter matrix — связи между числовыми переменными
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
pairs = [
    ('flipper_length_mm', 'body_mass_g'),
    ('bill_length_mm', 'bill_depth_mm'),
    ('bill_length_mm', 'flipper_length_mm'),
    ('bill_depth_mm', 'body_mass_g'),
    ('bill_length_mm', 'body_mass_g'),
    ('flipper_length_mm', 'bill_depth_mm'),
]
colors = {'Adelie': 'steelblue', 'Chinstrap': 'coral', 'Gentoo': 'green'}

for ax, (x, y) in zip(axes.flatten(), pairs):
    for species, grp in df_clean.groupby('species'):
        ax.scatter(grp[x], grp[y], label=species,
                   color=colors[species], alpha=0.5, s=20)
    ax.set_xlabel(x, fontsize=8)
    ax.set_ylabel(y, fontsize=8)
    ax.set_title(f'{x} vs {y}', fontsize=9)

axes[0, 0].legend(fontsize=8)
plt.suptitle('Scatter: числовые переменные по видам', fontsize=13)
plt.tight_layout()
plt.show()

**Ключевые корреляции (Spearman):**
- `flipper_length_mm` ↔ `body_mass_g` = ~0.87 — сильнейшая связь: большие плавники = тяжёлый пингвин
- `bill_length_mm` ↔ `flipper_length_mm` = ~0.66 — умеренная связь
- `bill_depth_mm` ↔ `bill_length_mm` = ~-0.23 — слабая обратная: Adelie (короткий клюв, глубокий) vs Chinstrap (длинный, мельче)
- `species_Gentoo` сильно коррелирует с `body_mass_g` — подтверждает что Gentoo = главный драйвер массы

## 6. Статистические тесты

In [ ]:
# Mann-Whitney: масса по полу
male = df_clean[df_clean['sex'] == 'male']['body_mass_g']
female = df_clean[df_clean['sex'] == 'female']['body_mass_g']
stat, p = mannwhitneyu(male, female)
print(f'Масса: male vs female — p = {p:.4f}  {"✅ значимо" if p < 0.05 else "❌ незначимо"}')
print(f'Медиана male:   {male.median():.0f}g')
print(f'Медиана female: {female.median():.0f}g')

print()

# Mann-Whitney: масса по островам
islands = df_clean['island'].unique()
print('Mann-Whitney: body_mass_g между островами')
print('-' * 50)
for i in range(len(islands)):
    for j in range(i+1, len(islands)):
        g1 = df_clean[df_clean['island'] == islands[i]]['body_mass_g']
        g2 = df_clean[df_clean['island'] == islands[j]]['body_mass_g']
        stat, p = mannwhitneyu(g1, g2)
        sig = '✅ значимо' if p < 0.05 else '❌ незначимо'
        print(f'{islands[i]:12} vs {islands[j]:12}: p = {p:.4f}  {sig}')

**Результаты тестов:**
- **Пол:** самцы значимо тяжелее самок (p < 0.05) ✅
- **Острова:** Biscoe значимо отличается от Torgersen и Dream (p=0.000)
- **Torgersen vs Dream:** незначимо (p=0.79) — оба содержат только Adelie, масса одинакова
- Mann-Whitney выбран т.к. не требует нормального распределения

## 7. Confounding variable: остров vs вид

In [ ]:
# Распределение видов по островам
print('Виды по островам:')
print(df_clean.groupby(['island', 'species']).size().unstack(fill_value=0))
print()
print('Медиана body_mass_g по островам:')
print(df_clean.groupby('island')['body_mass_g'].median().sort_values(ascending=False))

**Вывод о confounding variable:**

Наивный вывод: *"Остров влияет на массу пингвинов"*

Правильный вывод: **остров — это прокси для вида**, а вид — реальный объясняющий фактор:
```
Остров → Вид → Масса тела
```
- **Biscoe** = единственный остров с Gentoo (самый тяжёлый вид ~5000g)
- **Torgersen** = только Adelie (~3700g)
- **Dream** = Adelie + Chinstrap (~3700g)

Именно поэтому Torgersen vs Dream незначимы (p=0.79) — оба острова содержат схожие виды.
Это классический пример spurious correlation в EDA.

## 8. Итоговые выводы

### Качество данных
- 344 строки, 9 столбцов, дубликатов нет
- Удалено 11 строк с пропусками (3.2%) — безопасно при таком объёме

### Ключевые находки
1. **Gentoo** — самый крупный вид: масса ~5000g, длина плавника ~217mm
2. **Самцы крупнее самок** по всем числовым признакам (p < 0.05)
3. **Сильнейшая корреляция:** flipper_length ↔ body_mass (Spearman ~0.87)
4. **Бимодальные распределения** bill_length и flipper_length объясняются наличием подвидов
5. **Остров — не причина** различий в массе: это confounding variable через вид

### Ограничения
- Датасет небольшой (333 строки после очистки) — выводы ориентировочные
- `year` не использован — возможна временная динамика популяции
- Для построения классификатора видов рекомендуется: bill_length + bill_depth + flipper_length